# Notebook 03 — Open-Ended Evaluation
**AITR_Lab02_RushikeshShirsath_LLMFactuality**

Evaluates Llama-3.1-8B-Instruct and Mistral-7B-Instruct-v0.3 on the **open-ended** format of PubMedQA.
Computes token F1 against gold `long_answer` field.
Also performs failure case analysis (20 lowest-F1 items).

**Pre-condition:** Notebook 01 must pass. Notebook 02 results should exist.

**Expected time:** ~2 hours per model-condition on Colab T4.

In [ ]:
# Cell 1: Setup
import sys, os, json, random, time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import Counter

sys.path.insert(0, '/content')

from src.data_loader import load_pubmedqa_test
from src.evaluate import token_f1, compute_openended_metrics
from src.prompt_templates import OPENENDED_TEMPLATE, format_context
from src.rag import build_bm25_index, retrieve

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

os.makedirs('results', exist_ok=True)
print(f'CUDA: {torch.cuda.is_available()} | Seed: {SEED}')

In [ ]:
# Cell 2: Configuration — change these to switch condition
MODEL_NAME = 'mistral'    # 'mistral' or 'llama'
USE_RAG    = False        # False or True

MODEL_IDS = {
    'llama':   'meta-llama/Llama-3.1-8B-Instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}
FALLBACK_IDS = {
    'llama':   'BioMistral/BioMistral-7B',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}
RAG_TAG = 'rag' if USE_RAG else 'norag'
print(f'Config: model={MODEL_NAME}, rag={USE_RAG}, tag={RAG_TAG}')

In [ ]:
# Cell 3: HuggingFace login (Llama only)
if MODEL_NAME == 'llama':
    from huggingface_hub import login
    HF_TOKEN = 'your_hf_token_here'  # REPLACE — do NOT commit this
    login(token=HF_TOKEN)
    print('Logged in.')
else:
    print('Mistral-7B — no login required.')

In [ ]:
# Cell 4: Load test data and model
test_items = load_pubmedqa_test(n=500)
print(f'Test items: {len(test_items)}')

hf_id = MODEL_IDS[MODEL_NAME]
print(f'Loading: {hf_id}')
try:
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForCausalLM.from_pretrained(
        hf_id, load_in_8bit=True, device_map='auto')
except Exception as e:
    print(f'Failed: {e}\nFallback: {FALLBACK_IDS[MODEL_NAME]}')
    hf_id = FALLBACK_IDS[MODEL_NAME]
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForCausalLM.from_pretrained(
        hf_id, load_in_8bit=True, device_map='auto')
model.eval()
print(f'Loaded: {hf_id}')

In [ ]:
# Cell 5: Build BM25 index (if RAG)
bm25, doc_texts = None, None
if USE_RAG:
    print('Building BM25 index...')
    bm25, doc_texts = build_bm25_index(n_test=500)
    print(f'BM25 ready: {len(doc_texts)} docs')
else:
    print('RAG disabled.')

In [ ]:
# Cell 6: Run open-ended evaluation
def generate_oe(prompt):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=1024).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def categorize_failure(pred, gold, f1):
    if f1 >= 0.3: return 'ok'
    if f1 >= 0.1: return 'near_miss'
    if len(pred.split()) < 5: return 'incomplete'
    gold_kw = set(gold.lower().split()[:10])
    pred_kw = set(pred.lower().split()[:20])
    if not gold_kw & pred_kw: return 'off_topic'
    return 'hallucination'

preds, golds, rows = [], [], []
start = time.time()

for i, item in enumerate(test_items):
    q       = item['question']
    ctx     = format_context(item.get('context', {}), max_words=400)
    gold_of = item['long_answer']

    if USE_RAG:
        retrieved = retrieve(bm25, doc_texts, q, top_k=1)
        ctx = (retrieved + ' ' + ctx)[:800]
    else:
        ctx = ctx[:800]

    prompt  = OPENENDED_TEMPLATE.format(question=q, context=ctx)
    pred_of = generate_oe(prompt)
    f1      = token_f1(pred_of, gold_of)
    cat     = categorize_failure(pred_of, gold_of, f1)

    preds.append(pred_of)
    golds.append(gold_of)
    rows.append({'item_id': i, 'question': q[:80], 'gold_of': gold_of[:150],
                 'pred_of': pred_of[:150], 'token_f1': f1, 'failure_cat': cat})

    if (i + 1) % 10 == 0:
        mean_f1 = sum(r['token_f1'] for r in rows) / len(rows)
        print(f'  [{i+1}/500] Running mean F1: {mean_f1:.4f} | {time.time()-start:.0f}s')

    if (i + 1) % 50 == 0:
        with open(f'results/ckpt_oe_{MODEL_NAME}_{RAG_TAG}_{i+1}.json', 'w') as f:
            json.dump(rows, f)

print('Evaluation complete.')

In [ ]:
# Cell 7: Final metrics and failure analysis
metrics = compute_openended_metrics(preds, golds)
print('\n' + '='*60)
print(f'  FINAL OPEN-ENDED RESULTS — {MODEL_NAME.upper()} | RAG={USE_RAG}')
print('='*60)
print(f'  Mean Token F1: {metrics["mean_token_f1"]:.4f}')
print(f'  Min Token F1:  {metrics["min_token_f1"]:.4f}')
print(f'  Max Token F1:  {metrics["max_token_f1"]:.4f}')
print(f'  N items:       {metrics["n_items"]}')
print('='*60)

df = pd.DataFrame(rows)

# Failure case analysis — 20 worst items
print('\n=== FAILURE CASE ANALYSIS (20 lowest Token F1) ===')
worst_20 = df.nsmallest(20, 'token_f1')[['item_id','token_f1','failure_cat','question','pred_of']]
print(worst_20.to_string(index=False))

print('\n=== FAILURE CATEGORY DISTRIBUTION ===')
fail_dist = df['failure_cat'].value_counts()
for cat, cnt in fail_dist.items():
    print(f'  {cat:15s}: {cnt:3d}  ({cnt/len(df)*100:.1f}%)')

# Save
out_path = f'results/openended_results_{MODEL_NAME}_{RAG_TAG}.csv'
df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

summary = {'model': MODEL_NAME, 'rag': RAG_TAG,
           **{k: v for k, v in metrics.items() if k != 'scores'},
           'failure_distribution': fail_dist.to_dict()}
with open(f'results/summary_oe_{MODEL_NAME}_{RAG_TAG}.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\n>>> Copy the FINAL OPEN-ENDED RESULTS into Table 1 of paper/main.tex')